# Day 3 — NumPy + Pandas


In [1]:
import numpy as np
import pandas as pd
import time

In [2]:
#vectorization

n = 1_000_000
numbers = np.arange(n)

# the "slow" way - plain python loop
start = time.time()
squared_loop = [x ** 2 for x in numbers]
loop_time = time.time() - start

# the "fast" way - vectorized numpy
start = time.time()
squared_vectorized = numbers ** 2
vector_time = time.time() - start

print(f"Loop time:       {loop_time:.4f} sec")
print(f"Vectorized time: {vector_time:.4f} sec")
print(f"Vectorized was ~{loop_time / vector_time:.0f}x faster")

Loop time:       0.2420 sec
Vectorized time: 0.0319 sec
Vectorized was ~8x faster


Ok that's a genuinely huge difference for the exact same result. Officially converting to vectorizing everything from now on.

Next: merge, groupby, and pivot. Made up two small tables — orders and customers 

In [2]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "customer_name": ["Abhi", "Rahul", "Ekta", "Tejas"],
    "city": ["Mumbai", "Pune", "Pune", "Delhi"],
})

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105, 106],
    "customer_id": [1, 2, 1, 3, 4, 2],
    "amount": [250, 400, 150, 600, 300, 100],
})

# merge - joining orders with customer info, like a SQL join
merged = orders.merge(customers, on="customer_id", how="left")
merged

,order_id,customer_id,amount,customer_name,city
0,101,1,250,Abhi,Mumbai
1,102,2,400,Rahul,Pune
2,103,1,150,Abhi,Mumbai
3,104,3,600,Ekta,Pune
4,105,4,300,Tejas,Delhi
5,106,2,100,Rahul,Pune


In [3]:
# groupby - total amount spent per customer
spend_per_customer = merged.groupby("customer_name")["amount"].sum().sort_values(ascending=False)
print(spend_per_customer)

# pivot table - total amount spent per city, per customer
pivot = merged.pivot_table(values="amount", index="city", columns="customer_name", aggfunc="sum", fill_value=0)
pivot

customer_name
Ekta     600
Rahul    500
Abhi     400
Tejas    300
Name: amount, dtype: int64


customer_name,Abhi,Ekta,Rahul,Tejas
city,,,,
Delhi,0,0,0,300
Mumbai,400,0,0,0
Pune,0,600,500,0


groupby gave one number per customer (easy), pivot gave a proper cross-tab (customer x city). Took a couple tries to remember `fill_value=0` so it doesn't leave NaNs for combinations that don't exist — small thing but saved some confusion.

Last for today: missing data. Added some intentional gaps to practice on.

In [4]:
messy = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "age": [25, np.nan, 34, 29, np.nan],
    "city": ["Pune", "Mumbai", np.nan, "Delhi", "Pune"],
    "spend": [250, 400, 150, np.nan, 300],
})

print("Missing values per column:")
print(messy.isna().sum())
print()

# option 1: drop rows with ANY missing value (only good if you can afford to lose rows)
dropped = messy.dropna()

# option 2: fill missing values sensibly instead of just dropping everything
filled = messy.copy()
filled["age"] = filled["age"].fillna(filled["age"].median())
filled["spend"] = filled["spend"].fillna(filled["spend"].median())
filled["city"] = filled["city"].fillna("Unknown")

print("After dropna() - rows left:", len(dropped))
print()
print("After sensible fillna():")
filled

Missing values per column:
customer_id    0
age            2
city           1
spend          1
dtype: int64

After dropna() - rows left: 1

After sensible fillna():


,customer_id,age,city,spend
0,1,25.0,Pune,250.0
1,2,29.0,Mumbai,400.0
2,3,34.0,Unknown,150.0
3,4,29.0,Delhi,275.0
4,5,29.0,Pune,300.0


Note to self: `dropna()` is easy but with only 5 rows it wiped out 3 of them — way too aggressive for small/real datasets. Filling with median (for numbers) and a placeholder like "Unknown" (for categories) kept all the rows and felt like the more realistic approach. Keeping this in mind for tomorrow's EDA.
